# long-route-planner: flight videos

# Imports

In [ ]:
import json
from dataclasses import replace

import cv2
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

from uavloc.data.google_maps import create_session, load_api_key, satellite_tile_template
from uavloc.data.satellite import GeoImage
from uavloc.data.tiles import ESRI_WORLD_IMAGERY, XyzTileSource, geo_mosaic
from uavloc.features.segmentation import class_probabilities, load_segmenter
from uavloc.geo import GeoBounds, LatLon
from uavloc.matching.registration import apply_shift, phase_shift
from uavloc.plotting.flight import TRACK_LABELS, TRACK_RGB
from uavloc.plotting.flight_video import FlightVideo, FrameContext, VideoConfig, draw_text
from uavloc.simulation.camera import CameraConfig, RasterCamera
from uavloc.simulation.route import Pose, sample_route
from uavloc.simulation.sequence import FlightOutcome, as_flight_result, load_flight_set
from uavloc.track import REPO_ROOT

# Params

In [ ]:
VIDEOS = [   # (route, arm, seed, from_km, to_km, step_stride, name)
    ("best", "every-5s", 1, 0.0, None, 4, "best-every-5s-1-full"),       # held to 13.7 km, the longest
    ("best", "planner", 1, 0.0, None, 4, "best-planner-1-full"),
    ("best", "every-5s", 1, 12.0, 15.5, 1, "best-every-5s-1-loss"),     # the lock loss, every pose
]
ROUTE_SET = "sim-flights"
PLAN_DIR = "pocs/long-route-planner/runs/plan"
FLY_DIR = "pocs/long-route-planner/runs/fly"
VIDEO_DIR = "pocs/long-route-planner/figures/videos"
TILE_CACHE = "data/derived/tile-cache"
ZOOM = 18
GOOGLE_KEY = "google-satellite"
CHUNK_M = 2000.0                  # as fly.ipynb: the same rasters the flights used
CHUNK_MARGIN_M = 300.0
MIN_SHIFT_PEAK = 15.0
MAX_SHIFT_PX = 30.0
SEG_CHECKPOINT = "pocs/mobilenet-lraspp-speed/model-lraspp-bn-satellite.pt"
MATCH_GSD_M = 0.385
DEVICE = "cpu"
VIDEO_FPS = 20.0
VIDEO_SIZE_PX = (1280, 720)
VIDEO_MAP_RADIUS_M = 160.0
VIDEO_VIEW_WIDTH_PX = 640
VIDEO_CODEC = "avc1"
ERROR_AXIS_M = (1.0, 5000.0)      # log y-range of the error panel
ERROR_GRID_M = (10, 100, 1000)

# Data

In [ ]:
set_config, _ = load_flight_set(ROUTE_SET)
planned = json.loads((REPO_ROOT / PLAN_DIR / "routes.json").read_text())
fly_dir, video_dir = REPO_ROOT / FLY_DIR, REPO_ROOT / VIDEO_DIR
video_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
def route_poses(name: str) -> list[Pose]:
    return sample_route([LatLon(*p) for p in planned["routes"][name]], set_config.speed_mps, set_config.step_s,
                        set_config.altitude_m)

In [ ]:
def chunk_slices(n_poses: int) -> list[slice]:
    poses_per_chunk = int(CHUNK_M / (set_config.speed_mps * set_config.step_s))
    return [slice(i, i + poses_per_chunk) for i in range(0, n_poses, poses_per_chunk)]

In [ ]:
api_key = load_api_key()
google = XyzTileSource(GOOGLE_KEY, satellite_tile_template(api_key, create_session(api_key)))

In [ ]:
def chunk_rasters(chunk: list[Pose]) -> tuple[GeoImage, GeoImage]:
    """fly.ipynb's chunk rasters: Esri map, Google drone view co-registered onto it when the peak is reliable."""
    bounds = GeoBounds.around([p.position for p in chunk], CHUNK_MARGIN_M)
    esri = geo_mosaic(ESRI_WORLD_IMAGERY, bounds, ZOOM, REPO_ROOT / TILE_CACHE)
    drone = geo_mosaic(google, bounds, ZOOM, REPO_ROOT / TILE_CACHE)
    dx, dy, ratio = phase_shift(esri.pixels.mean(2), drone.pixels.astype(np.float32).mean(2))
    if ratio >= MIN_SHIFT_PEAK and np.hypot(dx, dy) <= MAX_SHIFT_PX:
        drone = drone.with_pixels(apply_shift(drone.pixels, dx, dy))
    return esri, drone

In [ ]:
def flight_result(route: str, arm: str, seed: int, poses: list[Pose]):
    track = pd.read_csv(fly_dir / f"track-{route}-{arm}-{seed}.csv")
    fixes = pd.read_csv(fly_dir / f"fixes-{route}-{arm}-{seed}.csv")
    return as_flight_result(FlightOutcome(None, track, fixes, float("nan")), poses, f"{arm} seed {seed}")

# Model

In [ ]:
drone_model = load_segmenter(REPO_ROOT / SEG_CHECKPOINT, DEVICE, "lraspp")

In [ ]:
def segment(view: Image.Image, width_m: float, height_m: float) -> np.ndarray:
    """Class ids of a drone view at the matcher's GSD, as the onboard segmenter sees it."""
    rgb = np.asarray(view.resize((round(width_m / MATCH_GSD_M), round(height_m / MATCH_GSD_M)), Image.BILINEAR))
    return class_probabilities(drone_model, rgb, DEVICE).argmax(0)

In [ ]:
def error_y_px(metres: np.ndarray, height: int) -> np.ndarray:
    low, high = np.log10(ERROR_AXIS_M)
    share = (np.log10(np.clip(metres, *ERROR_AXIS_M)) - low) / (high - low)
    return np.round((1 - share) * (height - 1)).astype(np.int32)

In [ ]:
def error_panel(ctx: FrameContext, size: tuple[int, int]) -> np.ndarray:
    """Corrected and INS-only error against distance, log scale; the flown part bright, a line at now."""
    width, height = size
    track = ctx.result.track
    x = np.round(track["t_s"].to_numpy() / track["t_s"].iloc[-1] * (width - 1)).astype(np.int32)
    panel = np.full((height, width, 3), 30, np.uint8)
    for metres in ERROR_GRID_M:
        y = int(error_y_px(np.array([metres]), height)[0])
        cv2.line(panel, (0, y), (width, y), (70, 70, 70), 1)
        draw_text(panel, [f"{metres} m"], (width - 70, max(y - 14, 0)), 0.35)
    for column, track_name in (("ins_error_m", "ins"), ("est_error_m", "estimate")):
        points = np.stack([x, error_y_px(track[column].to_numpy(), height)], 1)
        cv2.polylines(panel, [points], False, tuple(c // 3 for c in TRACK_RGB[track_name]), 1, cv2.LINE_AA)
        cv2.polylines(panel, [points[:ctx.step + 1]], False, TRACK_RGB[track_name], 2, cv2.LINE_AA)
    cv2.line(panel, (int(x[ctx.step]), 0), (int(x[ctx.step]), height), (255, 255, 255), 1)
    for k, track_name in enumerate(("truth", "estimate", "ins")):   # the map's legend, moved off the HUD
        draw_text(panel, [TRACK_LABELS[track_name]], (60 + 90 * k, 2), 0.4, TRACK_RGB[track_name])
    return panel

In [ ]:
video_config = VideoConfig(fps=VIDEO_FPS, frame_size_px=VIDEO_SIZE_PX, main_panel="map",
                           corner_panels={"top-right": "drone", "bottom-right": "segmentation", "top-left": "error"},
                           map_mode="follow", map_radius_m=VIDEO_MAP_RADIUS_M, segmentation_source="every-frame",
                           show_legend=False, codec=VIDEO_CODEC)
camera_config = replace(CameraConfig(set_config.fov_w_deg), image_width_px=VIDEO_VIEW_WIDTH_PX)

# Run

In [ ]:
def render(route: str, arm: str, seed: int, from_km: float, to_km: float | None, stride: int, name: str) -> str:
    """One mp4; the camera's raster and the map behind the tracks change at each 2 km chunk, as in the flight."""
    poses = route_poses(route)
    km = np.array([p.t_s for p in poses]) * set_config.speed_mps / 1e3
    first, last = int(np.searchsorted(km, from_km)), len(poses) if to_km is None else int(np.searchsorted(km, to_km))
    steps = set(range(first, last, stride))
    path = video_dir / f"{name}.mp4"
    writer = cv2.VideoWriter(str(path), cv2.VideoWriter_fourcc(*VIDEO_CODEC), VIDEO_FPS, VIDEO_SIZE_PX)
    video = None
    for part in tqdm(chunk_slices(len(poses)), desc=name):
        wanted = [s for s in range(part.start, min(part.stop, len(poses))) if s in steps]
        if not wanted:
            continue
        esri, drone = chunk_rasters(poses[part])
        camera = RasterCamera(drone, camera_config)
        if video is None:
            video = FlightVideo(flight_result(route, arm, seed, poses), camera, esri, video_config, segment,
                                {"error": error_panel})
        video.camera, video.map_raster = camera, esri
        for step in wanted:
            writer.write(cv2.cvtColor(video.frame(step), cv2.COLOR_RGB2BGR))
    writer.release()
    return str(path.relative_to(REPO_ROOT))

In [ ]:
made = [render(*spec) for spec in VIDEOS]
made